In [1]:
import pandas as pd
import numpy as np
import re
import os
from collections import Counter

# Load the clean 1,000 record sample from profiling step
df = pd.read_csv("open_patients_1000_clean.csv")

# Re-extract source type from _id
def get_source(id_str):
    if id_str.startswith('pmc-'):
        return 'pmc'
    elif id_str.startswith('usmle-'):
        return 'usmle'
    elif id_str.startswith('trec-ct-'):
        return 'trec-ct'
    elif id_str.startswith('trec-cds-'):
        return 'trec-cds'
    return 'unknown'

if 'source' not in df.columns:
    df['source'] = df['_id'].apply(get_source)

# Store original for before/after comparison
df['description_original'] = df['description'].copy()
df['word_count_before'] = df['description'].apply(lambda x: len(str(x).split()))

print(f"Loaded {len(df)} records")
print(f"Source distribution: {df['source'].value_counts().to_dict()}")
print(f"\nWord count stats BEFORE preprocessing:")
print(df['word_count_before'].describe().round(1))

Loaded 1000 records
Source distribution: {'pmc': 729, 'trec-ct': 125, 'trec-cds': 90, 'usmle': 56}

Word count stats BEFORE preprocessing:
count    1000.0
mean      339.4
std       253.3
min        14.0
25%       135.0
50%       283.0
75%       471.2
max      1910.0
Name: word_count_before, dtype: float64


In [2]:
# First, inspect USMLE endings to understand patterns
usmle_mask = df['source'] == 'usmle'
print(f"USMLE records: {usmle_mask.sum()}")
print("\n" + "="*80)
print("LAST 250 CHARS OF 8 SAMPLE USMLE RECORDS:")
print("="*80)

for idx, row in df[usmle_mask].head(8).iterrows():
    text = row['description']
    print(f"\n[{row['_id']}] ({len(text.split())} words)")
    print(f"  ...{text[-250:]}")
    # Show the actual last 3 characters for debugging
    print(f"  Last 3 chars: {repr(text[-3:])}")
    print("-"*40)

USMLE records: 56

LAST 250 CHARS OF 8 SAMPLE USMLE RECORDS:

[usmle-9392] (116 words)
  ...ation is recorded. The data gathered from the study are given in the table below:
Time point Total cases of lung cancer
t = 0 years 100
t = 5 years 500
t = 10 years 600
Which of the following is the incidence of lung cancer per 1,000 people per year?
  Last 3 chars: 'ar?'
----------------------------------------

[usmle-9119] (123 words)
  ...on, or recreational drug use. He is afebrile, and his vital signs are within normal limits. A physical examination is unremarkable. A urinalysis is normal. Which of the following medications was this patient most likely prescribed for his depression?
  Last 3 chars: 'on?'
----------------------------------------

[usmle-8] (129 words)
  ...se maculopapular rash, and bilateral flank pain. The remainder of the examination shows no abnormalities. Urinalysis shows:
Blood +3
Protein +1
RBC 10–12/hpf
RBC cast negative
Eosinophils numerous
Which of the following i

In [3]:
def strip_usmle_trailing_question(text):
    """
    Remove trailing diagnostic question from USMLE vignettes.
    
    v2 fixes:
    - Handles trailing quotes/punctuation after ? (e.g., ?", ?), ?')
    - Only removes the actual question sentence, not preceding data tables
    - Finds the question by locating the last '?' then walking back to 
      find where the question sentence begins
    """
    if not isinstance(text, str):
        return text
    
    text = text.strip()
    
    # Strip trailing quotes/whitespace to expose the '?'
    stripped_end = text.rstrip(' \'\"\)\]')
    
    if not stripped_end.endswith('?'):
        return text  # No trailing question found
    
    # Find the position of the last '?'
    last_q = stripped_end.rfind('?')
    if last_q == -1:
        return text
    
    # Walk backwards from the '?' to find where this question sentence starts.
    # USMLE questions typically start with:
    #   "Which of the following..."
    #   "What is the most..."
    #   "What pathological..."
    #   "How would you..."
    # We look for common question starters, or fall back to the last sentence boundary.
    
    question_starters = [
        'Which of the following',
        'What is the most',
        'What is the',
        'What pathological',
        'What would be',
        'What type of',
        'What condition',
        'What mechanism',
        'What drug',
        'What vitamin',
        'What enzyme',
        'What is this',
        'What should be',
        'What does this',
        'How would',
        'How does',
        'Where is the',
        'Where does',
    ]
    
    # Try to find a known question starter before the '?'
    question_start = -1
    for starter in question_starters:
        pos = text.rfind(starter)
        if pos != -1 and pos < last_q:
            question_start = pos
            break
    
    # If no known starter found, fall back: find the last sentence-ending
    # punctuation (.!?) BEFORE this question, then take everything after it
    if question_start == -1:
        # Search backwards from the '?' for the previous sentence end
        search_region = text[:last_q]
        # Find last '.', '!', or '?' before our trailing question
        for i in range(len(search_region) - 1, -1, -1):
            if search_region[i] in '.!?':
                question_start = i + 1
                break
    
    if question_start == -1:
        # Couldn't isolate the question — return original to be safe
        return text
    
    # Remove the question, keep everything before it
    cleaned = text[:question_start].strip()
    
    # Safety: don't remove more than 40% of the text
    if len(cleaned) < len(text) * 0.6:
        return text  # Stripping too much, something went wrong, keep original
    
    return cleaned


# Apply ONLY to USMLE records
df.loc[usmle_mask, 'description'] = df.loc[usmle_mask, 'description'].apply(
    strip_usmle_trailing_question
)

# Verify: show before/after for 5 records
print("BEFORE vs AFTER — USMLE Question Stripping")
print("="*80)

shown = 0
for idx, row in df[usmle_mask].iterrows():
    if shown >= 5:
        break
    original = row['description_original']
    cleaned = row['description']
    
    if original != cleaned:
        removed = original[len(cleaned):].strip()
        print(f"\n[{row['_id']}]")
        print(f"  REMOVED: \"{removed[:120]}{'...' if len(removed)>120 else ''}\"")
        print(f"  Words: {len(original.split())} → {len(cleaned.split())} (removed {len(original.split())-len(cleaned.split())})")
        shown += 1
    print("-"*40)

BEFORE vs AFTER — USMLE Question Stripping

[usmle-9392]
  REMOVED: "Which of the following is the incidence of lung cancer per 1,000 people per year?"
  Words: 116 → 101 (removed 15)
----------------------------------------

[usmle-9119]
  REMOVED: "Which of the following medications was this patient most likely prescribed for his depression?"
  Words: 123 → 109 (removed 14)
----------------------------------------

[usmle-8]
  REMOVED: "Which of the following is the most likely diagnosis?""
  Words: 129 → 120 (removed 9)
----------------------------------------

[usmle-8492]
  REMOVED: "Which of the following is the mechanism of this patient's pathology?"
  Words: 111 → 100 (removed 11)
----------------------------------------

[usmle-653]
  REMOVED: "Which of the following is the most appropriate next step in management?""
  Words: 182 → 170 (removed 12)
----------------------------------------


In [4]:
# Summary stats
usmle_df = df[usmle_mask].copy()
usmle_df['changed'] = usmle_df['description'] != usmle_df['description_original']
usmle_df['words_removed'] = (
    usmle_df['description_original'].apply(lambda x: len(str(x).split())) -
    usmle_df['description'].apply(lambda x: len(str(x).split()))
)

print(f"USMLE Question Removal Summary:")
print(f"  Records with questions stripped: {usmle_df['changed'].sum()} / {len(usmle_df)}")
print(f"  Records unchanged: {(~usmle_df['changed']).sum()}")
if usmle_df['changed'].any():
    print(f"  Avg words removed: {usmle_df.loc[usmle_df['changed'], 'words_removed'].mean():.1f}")
    print(f"  Max words removed: {usmle_df['words_removed'].max()}")
    print(f"  Min words removed: {usmle_df.loc[usmle_df['changed'], 'words_removed'].min()}")

# Flag any that removed too much (> 30 words) for manual review
heavy_strip = usmle_df[usmle_df['words_removed'] > 30]
if len(heavy_strip) > 0:
    print(f"\n⚠ {len(heavy_strip)} records had >30 words removed — review these:")
    for _, row in heavy_strip.iterrows():
        print(f"  {row['_id']}: {row['words_removed']} words removed")

USMLE Question Removal Summary:
  Records with questions stripped: 56 / 56
  Records unchanged: 0
  Avg words removed: 13.0
  Max words removed: 22
  Min words removed: 6


In [5]:
# ============================================================
# SAFE ABBREVIATIONS ONLY
# Criteria: 4+ characters, OR 3 characters that are medically
# unambiguous (no common English word matches)
# ============================================================

SAFE_ABBREVIATIONS = {
    # Conditions (4+ chars or unique 3-char)
    'HTN':  'hypertension',
    'T2DM': 'type 2 diabetes mellitus',
    'T1DM': 'type 1 diabetes mellitus',
    'COPD': 'chronic obstructive pulmonary disease',
    'CHF':  'congestive heart failure',
    'CAD':  'coronary artery disease',
    'CKD':  'chronic kidney disease',
    'ESRD': 'end-stage renal disease',
    'AML':  'acute myeloid leukemia',
    'GERD': 'gastroesophageal reflux disease',
    'DVT':  'deep vein thrombosis',
    'CVA':  'cerebrovascular accident',
    'TIA':  'transient ischemic attack',
    'UTI':  'urinary tract infection',
    'ARDS': 'acute respiratory distress syndrome',
    'SLE':  'systemic lupus erythematosus',
    'OSA':  'obstructive sleep apnea',
    'BPH':  'benign prostatic hyperplasia',
    'AFib': 'atrial fibrillation',
    
    # Symptoms (unique enough)
    'SOB':  'shortness of breath',
    'DOE':  'dyspnea on exertion',
    'LOC':  'loss of consciousness',
    
    # Medications
    'ASA':    'aspirin',
    'NSAIDs': 'nonsteroidal anti-inflammatory drugs',
    'NSAIDS': 'nonsteroidal anti-inflammatory drugs',
    'HCTZ':   'hydrochlorothiazide',
    
    # Procedures / Tests (unique enough)
    'CXR':  'chest X-ray',
    'EKG':  'electrocardiogram',
    'ECG':  'electrocardiogram',
    'EEG':  'electroencephalogram',
    'EMG':  'electromyography',
    'ECHO': 'echocardiogram',
    'ERCP': 'endoscopic retrograde cholangiopancreatography',
    'CABG': 'coronary artery bypass graft',
    'PCI':  'percutaneous coronary intervention',
}

# REMOVED (too ambiguous — let NER handle these in context):
# CT  (636 matches — "computed tomography" but matches everywhere)
# ALL (matches English word "all")
# RA  (matches partial words)
# AF  (2 letters, ambiguous)
# PE  (2 letters, ambiguous)
# MI  (2 letters, ambiguous)
# DM  (2 letters, ambiguous)
# HA  (2 letters, ambiguous)
# LP  (2 letters, ambiguous)
# PPI (matches non-medical contexts)
# N/V (slash causes regex issues)
# ARB (3 letters, common word)

print(f"Safe abbreviations in dictionary: {len(SAFE_ABBREVIATIONS)}")
print(f"Removed (ambiguous): CT, ALL, RA, AF, PE, MI, DM, HA, LP, PPI, N/V, ARB")

Safe abbreviations in dictionary: 35
Removed (ambiguous): CT, ALL, RA, AF, PE, MI, DM, HA, LP, PPI, N/V, ARB


In [6]:
# Scan: how many times do our SAFE abbreviations appear?
all_text = ' '.join(df['description'].astype(str))

print("Safe abbreviation frequency in our 1,000 records:")
print("="*60)

abbrev_counts = {}
for abbr in SAFE_ABBREVIATIONS:
    pattern = r'\b' + re.escape(abbr) + r'\b'
    count = len(re.findall(pattern, all_text))
    if count > 0:
        abbrev_counts[abbr] = count

total = 0
for abbr, count in sorted(abbrev_counts.items(), key=lambda x: -x[1]):
    print(f"  {abbr:8s} → {SAFE_ABBREVIATIONS[abbr]:45s} | {count}")
    total += count

print(f"\nTotal: {len(abbrev_counts)} types, {total} occurrences")

Safe abbreviation frequency in our 1,000 records:
  ECG      → electrocardiogram                             | 55
  EEG      → electroencephalogram                          | 20
  ERCP     → endoscopic retrograde cholangiopancreatography | 20
  CABG     → coronary artery bypass graft                  | 18
  ARDS     → acute respiratory distress syndrome           | 16
  CAD      → coronary artery disease                       | 15
  CXR      → chest X-ray                                   | 14
  HTN      → hypertension                                  | 13
  COPD     → chronic obstructive pulmonary disease         | 13
  DVT      → deep vein thrombosis                          | 13
  SLE      → systemic lupus erythematosus                  | 13
  EKG      → electrocardiogram                             | 12
  AML      → acute myeloid leukemia                        | 11
  CHF      → congestive heart failure                      | 9
  SOB      → shortness of breath                      

In [7]:
def expand_abbreviations_v2(text, abbrev_dict):
    """
    Expand medical abbreviations with safeguards.
    
    v2 fixes:
    - Skips expansion if the full form already appears within 5 words
      of the abbreviation (prevents double expansion)
    - Case-sensitive matching (abbreviations are uppercase)
    - Sorts by length descending to avoid overlap issues
    
    Format: "HTN" → "hypertension (HTN)" 
    This keeps the abbreviation visible for traceability while giving
    the NER model the full term to work with.
    """
    if not isinstance(text, str):
        return text, []
    
    expansions = []
    result = text
    
    sorted_abbrevs = sorted(abbrev_dict.keys(), key=len, reverse=True)
    
    for abbr in sorted_abbrevs:
        expansion = abbrev_dict[abbr]
        pattern = r'\b' + re.escape(abbr) + r'\b'
        
        matches = list(re.finditer(pattern, result))
        if not matches:
            continue
        
        # Process matches in REVERSE order so positions don't shift
        for match in reversed(matches):
            start, end = match.start(), match.end()
            
            # CHECK: Is the full expansion already nearby?
            # Look at a window around the abbreviation
            window_start = max(0, start - len(expansion) - 20)
            window_end = min(len(result), end + len(expansion) + 20)
            surrounding = result[window_start:window_end].lower()
            
            if expansion.lower() in surrounding:
                # Full form already present nearby — skip this one
                continue
            
            # CHECK: Is this inside parentheses? e.g., "(CAD)" 
            # If so, the full form is likely right before it
            if start > 0 and result[start-1] == '(' and end < len(result) and result[end] == ')':
                # Abbreviation is in parens like "(CAD)" — likely already explained
                continue
            
            # Safe to expand
            replacement = f"{expansion} ({abbr})"
            result = result[:start] + replacement + result[end:]
            expansions.append(f"{abbr} → {expansion}")
    
    return result, expansions


# Apply expansion
expansion_log = []

for idx in df.index:
    text = df.at[idx, 'description']
    expanded, exps = expand_abbreviations_v2(text, SAFE_ABBREVIATIONS)
    df.at[idx, 'description'] = expanded
    if exps:
        expansion_log.append({
            '_id': df.at[idx, '_id'],
            'source': df.at[idx, 'source'],
            'expansions': exps,
            'count': len(exps)
        })

print(f"Abbreviation Expansion Summary (v2):")
print(f"  Records with expansions: {len(expansion_log)} / {len(df)}")
print(f"  Total expansions made: {sum(e['count'] for e in expansion_log)}")

exp_df = pd.DataFrame(expansion_log) if expansion_log else pd.DataFrame()
if len(exp_df) > 0:
    print(f"\n  Per-source breakdown:")
    print(exp_df.groupby('source')['count'].agg(['count', 'sum', 'mean']).rename(
        columns={'count': 'records_affected', 'sum': 'total_expansions', 'mean': 'avg_per_record'}
    ).round(1))

Abbreviation Expansion Summary (v2):
  Records with expansions: 120 / 1000
  Total expansions made: 246

  Per-source breakdown:
          records_affected  total_expansions  avg_per_record
source                                                      
pmc                     83               160             1.9
trec-cds                15                33             2.2
trec-ct                 20                51             2.6
usmle                    2                 2             1.0


In [8]:
# Verify: show before/after examples
print("BEFORE vs AFTER — Abbreviation Expansion Examples (v2)")
print("="*80)

shown = 0
for log_entry in expansion_log:
    if shown >= 3:
        break
    if log_entry['count'] >= 2:
        _id = log_entry['_id']
        row = df[df['_id'] == _id].iloc[0]
        orig_row = df[df['_id'] == _id].iloc[0]
        
        print(f"\n[{_id}] ({row['source']})")
        print(f"  Expansions: {log_entry['expansions']}")
        
        # Show a snippet around one expansion
        expanded_term = log_entry['expansions'][0].split(' → ')[1]
        pos = row['description'].find(expanded_term)
        if pos >= 0:
            start = max(0, pos - 30)
            end = min(len(row['description']), pos + 70)
            snippet = row['description'][start:end]
            print(f"  Snippet: \"...{snippet}...\"")
            
            # Verify no double expansion
            if f"{expanded_term} ({expanded_term}" in row['description']:
                print(f"  ⚠ DOUBLE EXPANSION DETECTED — BUG!")
            else:
                print(f"  ✓ No double expansion")
        shown += 1
        print("-"*40)

# Global double-expansion check
double_exp_count = 0
for _, row in df.iterrows():
    for abbr, expansion in SAFE_ABBREVIATIONS.items():
        bad_pattern = f"{expansion} ({expansion}"
        if bad_pattern.lower() in row['description'].lower():
            double_exp_count += 1
            break

print(f"\n\n{'='*60}")
print(f"Global double-expansion check: {double_exp_count} records with doubles")
if double_exp_count == 0:
    print("✓ Double expansion bug is FIXED")
else:
    print("⚠ STILL HAS DOUBLE EXPANSIONS — investigate!")

BEFORE vs AFTER — Abbreviation Expansion Examples (v2)

[trec-cds-2016-3] (trec-cds)
  Expansions: ['CAD → coronary artery disease', 'CKD → chronic kidney disease']
  Snippet: "...x significant for severe PVD, coronary artery disease (CAD), DM, and chronic kidney disease (CKD) pr..."
  ✓ No double expansion
----------------------------------------

[trec-cds-2016-4] (trec-cds)
  Expansions: ['CAD → coronary artery disease', 'ECG → electrocardiogram']
  Snippet: "...rosis, multiple recent falls, coronary artery disease (CAD), who presents from nursing home with C2 ..."
  ✓ No double expansion
----------------------------------------

[trec-cds-2016-6] (trec-cds)
  Expansions: ['CAD → coronary artery disease', 'DVT → deep vein thrombosis']
  Snippet: "...s (DVT), atrial fibrillation, coronary artery disease (CAD) presents with fever and abdominal pain. ..."
  ✓ No double expansion
----------------------------------------


Global double-expansion check: 0 records with doubles
✓ Double ex

In [9]:
def clean_clinical_text(text):
    """
    Light text cleaning for clinical notes.
    
    DO clean: whitespace, unicode artifacts
    DON'T touch: case, punctuation, numbers, medical special chars
    """
    if not isinstance(text, str):
        return text
    
    # Normalize unicode
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    text = text.replace('\xa0', ' ')
    
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text


df['description'] = df['description'].apply(clean_clinical_text)
print(f"Text cleaning applied to {len(df)} records")

Text cleaning applied to 1000 records


In [10]:
def estimate_tokens(text):
    """Approximate tokens: words × 1.3"""
    return int(len(str(text).split()) * 1.3)


def split_into_sentences(text):
    """Split clinical text into sentences, protecting medical abbreviations."""
    protected = text
    abbrev_patterns = [
        r'\bDr\.', r'\bMr\.', r'\bMrs\.', r'\bMs\.', r'\bvs\.',
        r'\betc\.', r'\be\.g\.', r'\bi\.e\.', r'\bpt\.',
        r'\bapprox\.', r'\bno\.', r'\bFig\.', r'\bfig\.',
    ]
    for pattern in abbrev_patterns:
        protected = re.sub(pattern, lambda m: m.group().replace('.', '<P>'), protected)
    
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', protected)
    sentences = [s.replace('<P>', '.') for s in sentences]
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences


def chunk_text(text, record_id, max_tokens=400, overlap_tokens=50):
    """
    Sentence-aware chunking with overlap.
    Returns list of chunk dicts linked to original record.
    """
    total_tokens = estimate_tokens(text)
    
    if total_tokens <= max_tokens:
        return [{
            'chunk_id': f"{record_id}_chunk_0",
            '_id': record_id,
            'chunk_index': 0,
            'total_chunks': 1,
            'text': text,
            'approx_tokens': total_tokens,
            'is_chunked': False
        }]
    
    sentences = split_into_sentences(text)
    chunks = []
    current_sentences = []
    current_tokens = 0
    
    for sent in sentences:
        sent_tokens = estimate_tokens(sent)
        
        if current_tokens + sent_tokens > max_tokens and current_sentences:
            chunks.append(' '.join(current_sentences))
            
            # Overlap: carry last N sentences fitting within overlap_tokens
            overlap_sents = []
            overlap_count = 0
            for s in reversed(current_sentences):
                s_tok = estimate_tokens(s)
                if overlap_count + s_tok <= overlap_tokens:
                    overlap_sents.insert(0, s)
                    overlap_count += s_tok
                else:
                    break
            
            current_sentences = overlap_sents + [sent]
            current_tokens = overlap_count + sent_tokens
        else:
            current_sentences.append(sent)
            current_tokens += sent_tokens
    
    if current_sentences:
        chunks.append(' '.join(current_sentences))
    
    total_chunks = len(chunks)
    return [{
        'chunk_id': f"{record_id}_chunk_{i}",
        '_id': record_id,
        'chunk_index': i,
        'total_chunks': total_chunks,
        'text': chunk,
        'approx_tokens': estimate_tokens(chunk),
        'is_chunked': True
    } for i, chunk in enumerate(chunks)]


# Apply chunking
all_chunks = []
for idx, row in df.iterrows():
    row_chunks = chunk_text(row['description'], row['_id'])
    for c in row_chunks:
        c['source'] = row['source']
    all_chunks.extend(row_chunks)

chunks_df = pd.DataFrame(all_chunks)

print(f"Chunking Summary:")
print(f"  Original records: {len(df)}")
print(f"  Total chunks: {len(chunks_df)}")
print(f"  Records chunked: {chunks_df[chunks_df['is_chunked']]['_id'].nunique()}")
print(f"  Records single chunk: {(~chunks_df['is_chunked']).sum()}")
print(f"\n  Per source:")
print(chunks_df.groupby('source').agg(
    chunks=('chunk_id', 'count'),
    chunked_records=('is_chunked', 'sum'),
    avg_tokens=('approx_tokens', 'mean')
).round(1))

Chunking Summary:
  Original records: 1000
  Total chunks: 1667
  Records chunked: 465
  Records single chunk: 535

  Per source:
          chunks  chunked_records  avg_tokens
source                                       
pmc         1396             1132       305.1
trec-cds      90                0       120.4
trec-ct      125                0       157.2
usmle         56                0       136.8


In [11]:
# Chunk distribution
chunks_per_record = chunks_df.groupby('_id')['chunk_index'].max() + 1
print(f"Chunks per record:")
print(chunks_per_record.value_counts().sort_index().to_string())
print(f"\nMax chunks: {chunks_per_record.max()} (record: {chunks_per_record.idxmax()})")
print(f"Chunks over 450 tokens: {(chunks_df['approx_tokens'] > 450).sum()}")
print(f"Chunks over 512 tokens: {(chunks_df['approx_tokens'] > 512).sum()}")

Chunks per record:
chunk_index
1    542
2    308
3    111
4     28
5      5
6      4
7      1
8      1

Max chunks: 8 (record: pmc-5576441-1)
Chunks over 450 tokens: 0
Chunks over 512 tokens: 0


In [12]:
print("PREPROCESSING VALIDATION")
print("="*60)

df['word_count_after'] = df['description'].apply(lambda x: len(str(x).split()))

# Data integrity
empty = df['description'].isna().sum() + (df['description'].str.strip() == '').sum()
short = (df['word_count_after'] < 10).sum()
print(f"✓ Empty descriptions: {empty}")
print(f"✓ Records under 10 words: {short}")
print(f"✓ Source distribution: {df['source'].value_counts().to_dict()}")

# Word counts
print(f"\nWord counts:")
print(f"  Before: mean={df['word_count_before'].mean():.0f}, median={df['word_count_before'].median():.0f}")
print(f"  After:  mean={df['word_count_after'].mean():.0f}, median={df['word_count_after'].median():.0f}")

# Chunking
print(f"\nChunks:")
print(f"  Total: {len(chunks_df)}")
print(f"  Token range: {chunks_df['approx_tokens'].min()} - {chunks_df['approx_tokens'].max()}")
print(f"  Over 512 tokens: {(chunks_df['approx_tokens'] > 512).sum()}")

# Double expansion check
doubles = 0
for _, row in df.iterrows():
    for abbr, exp in SAFE_ABBREVIATIONS.items():
        if f"{exp} ({exp}" in row['description'].lower():
            doubles += 1
            break
print(f"\nDouble expansions: {doubles}")

print(f"\n{'='*60}")
if empty == 0 and short == 0 and doubles == 0:
    print("✓ ALL CHECKS PASSED — Ready for NER extraction")
else:
    print("⚠ Issues found — review above")

PREPROCESSING VALIDATION
✓ Empty descriptions: 0
✓ Records under 10 words: 0
✓ Source distribution: {'pmc': 729, 'trec-ct': 125, 'trec-cds': 90, 'usmle': 56}

Word counts:
  Before: mean=339, median=283
  After:  mean=339, median=283

Chunks:
  Total: 1667
  Token range: 18 - 410
  Over 512 tokens: 0

Double expansions: 0

✓ ALL CHECKS PASSED — Ready for NER extraction


In [13]:
# Save preprocessed full records
export_cols = ['_id', 'source', 'description']
df[export_cols].to_csv('open_patients_1000_preprocessed.csv', index=False)
print(f"Saved: open_patients_1000_preprocessed.csv")
print(f"  Records: {len(df)}")
print(f"  Size: {os.path.getsize('open_patients_1000_preprocessed.csv') / 1024:.1f} KB")

# Save chunks for NER pipeline
chunk_cols = ['chunk_id', '_id', 'source', 'chunk_index', 'total_chunks',
              'text', 'approx_tokens', 'is_chunked']
chunks_df[chunk_cols].to_csv('open_patients_chunks.csv', index=False)
print(f"\nSaved: open_patients_chunks.csv")
print(f"  Chunks: {len(chunks_df)}")
print(f"  Size: {os.path.getsize('open_patients_chunks.csv') / 1024:.1f} KB")

# Save expansion log
if expansion_log:
    pd.DataFrame(expansion_log).to_csv('preprocessing_abbreviation_log.csv', index=False)
    print(f"\nSaved: preprocessing_abbreviation_log.csv")

Saved: open_patients_1000_preprocessed.csv
  Records: 1000
  Size: 2203.2 KB

Saved: open_patients_chunks.csv
  Chunks: 1667
  Size: 2393.1 KB

Saved: preprocessing_abbreviation_log.csv
